# Crawling PTA UTM

In [ ]:
!pip install builtwith

  Preparing metadata (setup.py) ... done
  Created wheel for builtwith: filename=builtwith-1.3.4-py3-none-any.whl size=36077 sha256=1742c6a72e0c9a9da3e8990cecd160d00a5808335bf01a4db0696afb0578c4dc
  Stored in directory: /root/.cache/pip/wheels/7f/2d/b2/606e3df914d4aeeab99c4a4e3e9a61673d2293c2e346db00c8
Successfully built builtwith


In [ ]:
!pip install requests

In [ ]:
!pip install beautifulsoup4

In [ ]:
import builtwith
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from IPython.display import display
from urllib.parse import urljoin, urlparse

# **2. Analisis Teknologi Web PTA Trunojoyo**

In [ ]:
res = builtwith.parse('https://pta.trunojoyo.ac.id/')
print(res)

{'web-servers': ['Nginx'], 'javascript-frameworks': ['jQuery', 'jQuery UI']}


# **3. Menampilkan ID Semua Prodi**

In [ ]:
def pta_IDprodi(url):
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes

        soup = BeautifulSoup(response.content, 'html.parser')

        # Ambil semua judul h1, h2, h3
        headings = soup.find_all(['h1', 'h2', 'h3'])
        for heading in headings:
            print(f"{heading.name}: {heading.get_text()}")

        # Ambil semua link
        links = soup.find_all('a', href=True)
        for link in links:
            href = link['href']
            text = link.get_text().strip()

            # Filter: hanya link yang ada '/c_search/byprod/'
            if "/c_search/byprod/" in href:
                print(f"URL: {href} | Teks: {text}")


    except requests.exceptions.RequestException as e:
        print(f"Terjadi kesalahan saat mengakses {url}: {e}")

# Gunakan fungsi
pta_IDprodi("https://pta.trunojoyo.ac.id/welcome/index/2")

h2: Daftar Karya Ilmiah
URL: https://pta.trunojoyo.ac.id/c_search/byprod/1 | Teks: Ilmu Hukum
URL: https://pta.trunojoyo.ac.id/c_search/byprod/24 | Teks: Magister Ilmu Hukum
URL: https://pta.trunojoyo.ac.id/c_search/byprod/2 | Teks: Teknologi Industri Pertanian
URL: https://pta.trunojoyo.ac.id/c_search/byprod/3 | Teks: Agribisnis
URL: https://pta.trunojoyo.ac.id/c_search/byprod/4 | Teks: Agroteknologi
URL: https://pta.trunojoyo.ac.id/c_search/byprod/5 | Teks: Ilmu Kelautan
URL: https://pta.trunojoyo.ac.id/c_search/byprod/35 | Teks: Manajemen Sumberdaya Perairan
URL: https://pta.trunojoyo.ac.id/c_search/byprod/37 | Teks: Magister Pengelolaan Sumber Daya Alam
URL: https://pta.trunojoyo.ac.id/c_search/byprod/6 | Teks: Ekonomi Pembangunan
URL: https://pta.trunojoyo.ac.id/c_search/byprod/7 | Teks: Manajemen
URL: https://pta.trunojoyo.ac.id/c_search/byprod/8 | Teks: Akuntansi
URL: https://pta.trunojoyo.ac.id/c_search/byprod/21 | Teks: D3 Akuntansi
URL: https://pta.trunojoyo.ac.id/c_search/by

# **4. Mengambil dan Menyimpan Data Karya Ilmiah**

In [ ]:
def pta_KaryaIlmiah():
    data = {
        "prodi_id": [],
        "penulis": [],
        "judul": [],
        "pembimbing_pertama": [],
        "pembimbing_kedua": [],
        "abstrak_bindonesia": [],
        "abstrak_binggris": []
    }

    # Loop semua id prodi (1 sampai 40)
    for prodi_id in range(1, 41):
        page = 1
        while True:
            url = f"https://pta.trunojoyo.ac.id/c_search/byprod/{prodi_id}/{page}"
            r = requests.get(url)
            soup = BeautifulSoup(r.content, "html.parser")
            jurnals = soup.select('li[data-cat="#luxury"]')

            # hentikan loop (jika tidak ada jurnal di halaman lain)
            if not jurnals:
                break

            for jurnal in jurnals:
                detail_url = jurnal.select_one('a.gray.button')['href']
                response = requests.get(detail_url)
                soup1 = BeautifulSoup(response.content, "html.parser")

                isi = soup1.select_one('div#content_journal')
                judul = isi.select_one('a.title').text.strip()

                penulis = isi.select_one('span:contains("Penulis")').text.split(' : ')[1].strip()
                pembimbing_pertama = isi.select_one('span:contains("Dosen Pembimbing I")').text.split(' : ')[1].strip()
                pembimbing_kedua = isi.select_one('span:contains("Dosen Pembimbing II")').text.split(' :')[1].strip()

                # Ambil semua paragraf abstrak
                abstrak_paragraf = isi.find_all('p', align="justify")
                abstrak_bindonesia = abstrak_paragraf[0].get_text(strip=True) if len(abstrak_paragraf) > 0 else ""
                abstrak_binggris   = abstrak_paragraf[1].get_text(strip=True) if len(abstrak_paragraf) > 1 else ""

                # Simpan ke dict
                data["prodi_id"].append(prodi_id)
                data["penulis"].append(penulis)
                data["judul"].append(judul)
                data["pembimbing_pertama"].append(pembimbing_pertama)
                data["pembimbing_kedua"].append(pembimbing_kedua)
                data["abstrak_bindonesia"].append(abstrak_bindonesia)
                data["abstrak_binggris"].append(abstrak_binggris)

            print(f"Selesai ambil data Prodi {prodi_id}, Halaman {page}")
            page += 1  # halaman berikutnya

    df = pd.DataFrame(data)
    df = df.drop(columns=["prodi_id"])  # hapus kolom prodi_id
    df.to_csv("PPW_HasilCrawling_Tugas2(PTATrunojoyo).csv", index=False, encoding="utf-8-sig")
    return df

In [ ]:
df = pta_KaryaIlmiah()

/usr/local/lib/python3.12/dist-packages/soupsieve/css_parser.py:876: FutureWarning: The pseudo class ':contains' is deprecated, ':-soup-contains' should be used moving forward.
  warnings.warn(  # noqa: B028


Selesai ambil data Prodi 1, Halaman 1
Selesai ambil data Prodi 1, Halaman 2
Selesai ambil data Prodi 1, Halaman 3
Selesai ambil data Prodi 1, Halaman 4
Selesai ambil data Prodi 1, Halaman 5
Selesai ambil data Prodi 1, Halaman 6
Selesai ambil data Prodi 1, Halaman 7
Selesai ambil data Prodi 1, Halaman 8
Selesai ambil data Prodi 1, Halaman 9
Selesai ambil data Prodi 1, Halaman 10
Selesai ambil data Prodi 1, Halaman 11
Selesai ambil data Prodi 1, Halaman 12
Selesai ambil data Prodi 1, Halaman 13
Selesai ambil data Prodi 1, Halaman 14
Selesai ambil data Prodi 1, Halaman 15
Selesai ambil data Prodi 1, Halaman 16
Selesai ambil data Prodi 1, Halaman 17
Selesai ambil data Prodi 1, Halaman 18
Selesai ambil data Prodi 1, Halaman 19
Selesai ambil data Prodi 1, Halaman 20
Selesai ambil data Prodi 1, Halaman 21
Selesai ambil data Prodi 1, Halaman 22
Selesai ambil data Prodi 1, Halaman 23
Selesai ambil data Prodi 1, Halaman 24
Selesai ambil data Prodi 1, Halaman 25
Selesai ambil data Prodi 1, Halama

In [ ]:
display(df.sample(10))

,penulis,judul,pembimbing_pertama,pembimbing_kedua,abstrak_bindonesia,abstrak_binggris
1441,Doris Angga Reko,ANALISIS DISTRIBUSI DAN TRANSPORTASI RANTAI PA...,"Iffan Maflahah, STP. MSi","Supriyanto, STP. MP",Doris Angga Reko. 07.03.3.1.1.0007 . ANALISIS ...,Doris Angga Reko. 07.03.3.1.1.0007. ANALYSIS O...
1007,KOHAR NURHAMIDIN,PERLINDUNGAN HUKUM TERHADAP TINDAK PENCURIAN D...,"TOLIB EFFENDI. S.H., M.H.",,Untuk Kemajuan teknologi dimasa sekarang ini ...,For the advancement of technology in the prese...
4859,MUSRIFAH,Penentuan Lokasi Usaha Berdasarkan Pendekatan ...,"Dr. Ir. Nurita Andriani.,M.M","Dr. Mohammad Arief, SE.,M.M","Musrifah (2018), Penentuan Lokasi Usaha Berdas...",Musrifah (2018) location determination based o...
2557,Jakfar Sodik,PENGARUH JUMLAH BIBIT TERHADAP PERTUMBUHAN DAN...,"Ir. H. Suhartono, MP","Ir. Sucipto, MP",ABSTRAK\r\n\tPenelitian ini bertujuan untuk me...,ABSTRACT\r\nThis study aimed to determine the ...
12815,Diana Wahyuningsih,Pengaruh Loyalty Program dan Mall Atmosphere T...,"Dr. H. Pribanus Wantara, Drs., M.M.","Dr. H. Muh. Syarif, Drs. Ec., M.Si.",Perkembangan zaman yang semakin modern diikuti...,The development of an increasingly modern era ...
10838,Qori Isnaini,ANALISIS PENERAPAN KONSEP BISNIS SYARIAH DI HO...,"Firman Setiawan S.HI., M.EI",,Penelitian ini dilatar belakangi karena masih ...,This study is a type of causal of relationship...
6040,bangga ardiansyah,ANALISIS LAPORAN KEUANGAN UNTUK MENILAI KINERJ...,alexander anggono,,Koperasi Kesejahteraan Keluarga Besar Univ Is...,Koperasi Kesejahteraan Keluarga Besar Univ Is...
13096,Luluk Hanifah,Analisis Pengendalian Internal Terhadap Penceg...,"Dr. Siti Musyarrofah, S.E.,M.Si.,CA","Dr. Prasetyono.,S.E.,M.Si.,Ak","ABSTRAK\r\n\r\nLuluk Hanifah, Analisis Pengend...",ABSTRACT\r\n\r\nThis research is addressed to ...
13160,Hanif Yusuf Seputro,ANALISIS EFEK MODERASI KOMITMEN ORGANISASI PAD...,"Prof. Dr. M. Nizarul Alim, SE.,M.Si., Ak., CA","Dr. Tarjo, SE.,M.Si., CPAI., CFE",Abstraksi: Tujuan penelitian ini adalah untuk ...,Abstraction: The purpose of this study was to ...
3241,Hendra Jaya Kusuma,ANALISA KANDUNGAN LOGAM BERAT Fe PADA AIR DAN ...,"Dr. H. Agus Romadho., S.P M.SI","Maulinna Kusumo Wardhani.,S.Kel., M.Si",ABSTRAK\r\n\r\nSecara alami unsur-unsur logam ...,ABSTRACT\r\n\tNaturally heavy metal elements c...
